In [ ]:
# !pip install python-dotenv
!pip install faiss-cpu
# !pip install scikit-learn
# !pip install torch
# !pip install nltk
# !pip uninstall -y huggingface_hub
!pip install sentence_transformers==2.2.2
!pip install huggingface_hub
!pip install openai
!pip install llama-index
!pip install langchain==0.1.13


In [ ]:
import json
from pathlib import Path
from typing import List, Dict, Optional

import pandas as pd
import numpy as np
from google.colab import drive
import os
import json
import os
import io
import re
#import requests
import dotenv
import transformers
# import pypdf
import faiss
import time
from sklearn.cluster import DBSCAN
import torch
import tiktoken
import random

#import sqlite3

from dotenv import load_dotenv
import openai
from openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
import huggingface_hub
# run for huggingface_hub==36.0
if not hasattr(huggingface_hub, "cached_download"):
    from huggingface_hub import hf_hub_download

    def cached_download(*args, **kwargs):
        return hf_hub_download(*args, **kwargs)

    huggingface_hub.cached_download = cached_download
from sentence_transformers import util, SentenceTransformer
from transformers import pipeline, BertTokenizer, BertModel

from io import StringIO
#from operator import itemgetter

import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')
nltk.download('punkt')

from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter, SemanticSplitterNodeParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Document

In [ ]:
from google.colab import drive
from google.colab import userdata

openai_api_key = userdata.get('OPEN_API_KEY')
if openai_api_key:
    os.environ['OPENAI_API_KEY'] = openai_api_key

hf_token = userdata.get('HF_TOKEN')
# Optional: use the Hugging Face token only in the runtime when needed.

drive.mount('/content/drive')


In [ ]:
# duplicate paths
def reviewPath(filename):
  return '/content/drive/MyDrive/datasets/cleaned_reviews_' + filename + '_single.parquet'
def descriptionPath(filename):
  return '/content/drive/MyDrive/datasets/top_product_' + filename + '_single.csv'

In [ ]:
CSJ_review = reviewPath("Clothing_Shoes_and_Jewelry")
CSJ_description = descriptionPath("Clothing_Shoes_and_Jewelry")
elec_review = reviewPath("Electronics")
elec_description = descriptionPath("Electronics")
HH_review = reviewPath("Health_and_Household")
HH_description = descriptionPath("Health_and_Household")

df_CSJ = pd.read_parquet(CSJ_review)
df_elec = pd.read_parquet(elec_review)
df_HH = pd.read_parquet(HH_review)
df_CSJ_desc = pd.read_csv(CSJ_description)
df_elec_desc = pd.read_csv(elec_description)
df_HH_desc = pd.read_csv(HH_description)

for df in (df_CSJ, df_elec, df_HH):
  df['merged_review'] = df['review_title'] + ' ' + df['text']

In [ ]:
df_CSJ_desc['title'][0]

# Description Analysis

In [ ]:
# 1. product description
product_description = df_CSJ_desc['product_description'][0]
# product_description = df_elec_desc['product_description'][0]
# product_description = df_HH_desc['product_description'][0]



# 2. Prompt template for extracting structured information
description_prompt = f"""
You are analyzing a product based on its official description.

Here is the product description:
---
{product_description}
---

Extract the following information in JSON format:

1. key_features: core product features being advertised
2. strengths: what the description claims the product does well
3. target_user_persona: inferred demographic and use cases
4. value_propositions: unique selling points
5. design_attributes: colors, materials, shapes, and visual cues
6. performance_dimensions: aspects that could be evaluated in reviews
7. potential_risks_or_ambiguities: areas where expectations may not match customer experience

Return ONLY valid JSON.
"""

# 3. Call the LLM
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a concise and precise product analyst."},
        {"role": "user", "content": description_prompt}
    ],
    temperature=1
)

In [ ]:
description_output = response.choices[0].message.content

In [ ]:
expand_prompt = f"""
You are given structured product information extracted from the official product description:

{json.dumps(description_output, indent=2)}

Your task now is to expand the key features into a richer, more detailed and more granular list.
Break each key feature into 2–3 sub-features covering:

- technical details
- visual characteristics
- user benefits
- functional implications
- design cues

Return the output in JSON as:
{{
  "expanded_key_features": [
    {{
      "feature": "...",
      "details": ["...", "...", "..."]
    }}
  ]
}}
"""


In [ ]:
response2 = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are an expert product analyst."},
        {"role": "user", "content": expand_prompt}   # or image_attr_prompt, or infer_prompt
    ],
    temperature=1
)

second_pass_output = response2.choices[0].message.content

try:
    second_pass_json = json.loads(second_pass_output)
    print(json.dumps(second_pass_json, indent=4))
except:
    print("Raw output:")
    print(second_pass_output)


In [ ]:
image_attr_prompt = f"""
Based on the structured product description below:

{json.dumps(description_output, indent=2)}

Extract and expand ONLY the visual and physical attributes that are relevant for image generation.
Include:

- shape & geometry
- colors
- materials
- textures
- cable characteristics
- ergonomic elements
- microphone/button details
- scale
- relative proportions
- any real-world cues implied by the performance or design attributes

Return JSON as:

{{
  "visual_attributes_for_image_generation": [
      "attribute 1",
      "attribute 2",
      "attribute 3",
      ...
  ]
}}
"""


In [ ]:
response3 = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are an expert product analyst."},
        {"role": "user", "content": image_attr_prompt}   # or image_attr_prompt, or infer_prompt
    ],
    temperature=0
)

second_pass_output = response3.choices[0].message.content

try:
    second_pass_json = json.loads(second_pass_output)
    print(json.dumps(second_pass_json, indent=4))
except:
    print("Raw output:")
    print(second_pass_output)


# Sentence Chunking

In [ ]:
CSJ_review_text = [Document(text=t) for t in df_CSJ['merged_review'].dropna().tolist()]
elec_review_text = [Document(text=t) for t in df_elec['merged_review'].dropna().tolist()]
HH_review_text = [Document(text=t) for t in df_HH['merged_review'].dropna().tolist()]

### 3 types of chunking:

#### **Sentence Splitter**

In [ ]:
sentence_splitter = TokenTextSplitter(
    chunk_size=1024,
    chunk_overlap=200
)

#### **Semantic Splitter**

In [ ]:
# from llama_index import HuggingFaceEmbedding
semantic_splitter = SemanticSplitterNodeParser(
    include_metadata = True,
    include_prev_next_rel = True,
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model = OpenAIEmbedding(model="text-embedding-3-small")
)

#### **Token Splitter**

In [ ]:
token_splitter = TokenTextSplitter(
    chunk_size=1024,
    chunk_overlap=200
)

### Get Notes

In [ ]:
def get_chuncks(method,text):
  match method:
    case "sentence":
      nodes = sentence_splitter.get_nodes_from_documents(text)
    case "semantic":
      nodes = semantic_splitter.get_nodes_from_documents(text)
    case "token":
      nodes = token_splitter.get_nodes_from_documents(text)
  return [node.get_content() for node in nodes]

In [ ]:
# method to choose: sentence, semantic, token
text_chunks = get_chuncks("semantic",CSJ_review_text)

In [ ]:
text_chunks[:10] # glance at the text chunks to observe how the chunks look.

In [ ]:
# get the length of the sentences dataframe:
len(text_chunks)

### **Choose an embedding model to use for creating embeddings of the text chunks and create the Embeddings**

In [ ]:
def chunk_list(input_list, chunk_size):
    """Yield successive n-sized chunks from input_list."""
    for i in range(0, len(input_list), chunk_size):
        yield input_list[i:i + chunk_size]

In [ ]:
# Determine a suitable chunk size. OpenAI's limit is 300,000 tokens.
batch_size = 1000
text_chunk_batches = list(chunk_list(text_chunks, batch_size))

print(f"Total text chunks: {len(text_chunks)}")
print(f"Number of batches: {len(text_chunk_batches)}")
print(f"Size of first batch: {len(text_chunk_batches[0])}")

In [ ]:
# Using OpenAI embeddings
client = OpenAI()
embeddings = []

for i, batch in enumerate(text_chunk_batches):
    print(f"Processing batch {i+1}/{len(text_chunk_batches)}")
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    embeddings.extend([d.embedding for d in response.data])

print(f"Total embeddings generated: {len(embeddings)}")

## **Create a FAISS Vector Store**

In [ ]:
# specify the dimensions of the sentence embeddings:
review_embed = np.array(embeddings).astype("float32")   # (num_chunks, 1536)
print(review_embed.shape)
dimension = review_embed.shape[1]
# create an index for the vector store:
index = faiss.IndexFlatL2(dimension)
# add the embeddings to the index:
index.add(review_embed)
print("Total vectors in index:", index.ntotal)

In [ ]:
# train the index:
index.train(review_embed)
index.is_trained  # check if index is now trained

### **Construct Query and Perform Search of the Vector Store**

In [ ]:
# define a function to get retrieve the results from the vector store:
def get_sys_message(user_query: str, retrieval_number: int):
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=[user_query]
        ).data[0].embedding
    query_embedding = np.array([query_embedding]).astype("float32")
    D, I = index.search(query_embedding, retrieval_number)  # search
    first_index = I[0]  # Get the first index from I
    first_row_string = "\n".join(text_chunks[i] for i in first_index)
    return first_row_string

In [ ]:
get_sys_message(user_query="which review discussed the sound quality?", retrieval_number=10)

1. avg tokens/ review
2. how many reviews we want to look at = n
3. randomly select n from ...
4. put n in {f}
5. prompt "what are the top 3 discussed topics?" -> prompt engineering

In [ ]:
enc = tiktoken.encoding_for_model("gpt-5-mini")

def count_tokens(text):
    return len(enc.encode(text))

def sample_reviews_by_token_budget(chunks, token_budget=200_000):
    reviews = chunks.copy()
    random.shuffle(reviews)

    selected = []
    total_tokens = 0

    for r in reviews:
        t = count_tokens(r)
        if total_tokens + t > token_budget:
            break
        selected.append(r)
        total_tokens += t

    return selected, total_tokens


In [ ]:
review_subset, total_tokens = sample_reviews_by_token_budget(text_chunks, token_budget=200_000)
print("Selected reviews:", len(review_subset))
print("Total tokens:", total_tokens)
subset_text = "\n\n---\n\n".join(review_subset)

In [ ]:
client = OpenAI()

prompt = f"""
Below is a collection of product reviews (sampled to ~200k tokens):

================= REVIEWS BEGIN =================
{subset_text}
================= REVIEWS END =================

Using ONLY the content from these reviews:

Identify the top 3 most discussed topics.
   - approx. how frequently it appeared (estimate %)

Return results in clean bullet-point text.
"""

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are an expert product review analyst."},
        {"role": "user", "content": prompt}
    ],
    temperature=1
)

print(response.choices[0].message.content)

In [ ]:
topics = ['scent','packaging','quality effectiveness']

### **Perform LLM with RAG**

In [ ]:
# custom function for using an LLM with a RAG retriever:
def rag_openAI_gpt(
    model: str,
    query: str,
    retrieval_number: int,
    llm_prompt: str):
  client = OpenAI()
  f = get_sys_message(query, retrieval_number)
  system_prompt = f"""
  Instruction: Below is a collection of product reviews:

  ================= REVIEWS BEGIN =================
  {f}
  ================= REVIEWS END =================

  Using ONLY the content from these reviews to answer the user's question.

  Response format: give a summarizing sentence, then elaborate in bullet points.
  """
  response = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": f"{system_prompt}"},
           {"role": "user", "content": f"{llm_prompt}"},
            {"role": "assistant", "content": f"{f}"},
             {"role": "user", "content": "What is the answer?"}
          ]
      )
  return response.choices[0].message.content

In [ ]:
# rag_openAI_gpt(model="gpt-5-mini", query="which review discussed packaging?", retrieval_number=1000, llm_prompt="summarize the review's opions on packaging")

In [ ]:
# model choice: "gpt-3.5-turbo", "gpt-4", "gpt-4-0125-preview", "gpt-4o"
def sentiment_synthesis(model,topics,k):
  result = ""
  for topic in topics:
    query  = f"which review discussed {topic}?"
    prompt = f"summarize the review's opions on {topic}"
    result += f"{topic}: {rag_openAI_gpt(model=model, query=query, retrieval_number=k, llm_prompt=prompt)}\n\n"
  return result

In [ ]:
# - Sound quality (clarity, bass, overall audio): ~70% of reviews
# - Fit / comfort / “stays in the ear” (ergonomic ErgoFit shape, multiple tip sizes): ~55% of reviews
# - Durability / longevity (wiring, shorting, one‑side failures, surviving washes): ~45% of reviews
print(sentiment_synthesis("gpt-5-mini",topics,1000))